# Sesión 12 · Redpanda → Delta Lake en Databricks

Notebook adaptado de Python_12.ipynb. Lee lotes JSON del topic aviationstack-topic,
los descompone en una fila por vuelo y guarda el resultado en workspace.bronze.vuelos.
Configura el ámbito de secretos y el volumen de Unity Catalog antes de ejecutar las celdas.


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("secret_scope", "aviationstack")
dbutils.widgets.text("checkpoint_path", "/Volumes/workspace/bronze/checkpoints/aviationstack-vuelos")
dbutils.widgets.text("usuario", "Gonzalo Gomez")

scope = dbutils.widgets.get("secret_scope")
bootstrap_servers = dbutils.secrets.get(scope=scope, key="kafka-bootstrap-servers")
username = dbutils.secrets.get(scope=scope, key="kafka-username")
password = dbutils.secrets.get(scope=scope, key="kafka-password")
topic = "aviationstack-topic"

catalogo = "workspace"
esquema = "bronze"
tabla = "vuelos"
tabla_destino = f"{catalogo}.{esquema}.{tabla}"
checkpoint_path = dbutils.widgets.get("checkpoint_path")
usuario = dbutils.widgets.get("usuario")


def jaas_quote(value):
    return '"' + value.replace(chr(92), chr(92) * 2).replace('"', chr(92) + '"') + '"'


jaas_config = (
    "kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required "
    f"username={jaas_quote(username)} password={jaas_quote(password)};"
)


In [ ]:
df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)

df_kafka.printSchema()


In [ ]:
schema = """
ARRAY<STRUCT<
    vuelo: STRING,
    aerolinea: STRING,
    fecha_vuelo: STRING,
    origen: STRING,
    destino: STRING,
    latitud: DOUBLE,
    longitud: DOUBLE,
    altitud: DOUBLE,
    velocidad: DOUBLE,
    timestamp: TIMESTAMP
>>
"""

df_bronze = (
    df_kafka.selectExpr(
        "CAST(value AS STRING) AS payload_raw",
        "topic AS kafka_topic",
        "partition AS kafka_partition",
        "offset AS kafka_offset",
        "timestamp AS kafka_timestamp"
    )
    .withColumn("data", F.from_json("payload_raw", schema))
    .selectExpr(
        "*",
        "posexplode(data) AS (kafka_posicion, vuelo_data)"
    )
    .select(
        "vuelo_data.*",
        "kafka_topic",
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "kafka_posicion",
        "payload_raw"
    )
    .withColumn("fecha_vuelo", F.to_timestamp("fecha_vuelo", "yyyy-MM-dd"))
    .withColumn("altitud", F.col("altitud").cast("int"))
    .withColumn("user", F.lit(usuario))
    .withColumn("fecha_ingesta", F.current_timestamp())
    .withColumn("id", F.expr("""
        sha2(concat_ws('|', kafka_topic, kafka_partition,
        kafka_offset, coalesce(kafka_posicion, -1)), 256)
    """))
)

In [ ]:
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(tabla_destino)
)

query.awaitTermination()

In [ ]:
display(
    spark.table(tabla_destino)
    .select("vuelo", "destino", "kafka_partition", "kafka_offset", "kafka_posicion", "fecha_ingesta")
    .orderBy(F.desc("fecha_ingesta"))
    .limit(20)
)
